# 📊 CCIP — Giai đoạn 1: Khảo sát & Hồ sơ hóa Dữ liệu

**Mục tiêu notebook này:**
1. ✅ Phát hiện môi trường (Colab vs Local) và cấu hình phù hợp
2. ✅ Tải dữ liệu Home Credit từ Kaggle
3. ✅ Data Profiling tự động bằng `ydata-profiling`
4. ✅ Khảo sát thủ công: missing values, outlier, phân phối
5. ✅ Ghi chú `data dictionary` ngay trong notebook

---
> **Chạy trên Google Colab**: Nhấn `Runtime → Run all` (hoặc `Ctrl+F9`)  
> **Chạy Local**: `jupyter notebook analysis/01_data_profiling.ipynb`

## 0️⃣ Phát hiện môi trường & Cài đặt thư viện

In [ ]:
import sys, os

# Phát hiện đang chạy trên Colab hay Local
IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'🔍 Môi trường: {"Google Colab" if IN_COLAB else "Local Jupyter"}')
print(f'🐍 Python: {sys.version.split()[0]}')

In [ ]:
# Cài thư viện — chỉ cần chạy 1 lần
# Trên Colab: tự động cài
# Trên Local: đã có trong requirements.txt, cell này sẽ skip nhanh

if IN_COLAB:
    print('📦 Cài thư viện cho Colab...')
    os.system('pip install -q ydata-profiling kaggle pandas numpy matplotlib seaborn')
    print('✅ Cài xong!')
else:
    print('ℹ️  Local: Đảm bảo đã chạy: pip install -r requirements.txt')

## 1️⃣ Xác thực Kaggle & Tải dữ liệu

In [ ]:
from pathlib import Path

if IN_COLAB:
    # ────────────────────────────────────────────────────────
    # COLAB: 2 cách xác thực Kaggle — chọn 1 trong 2
    # ────────────────────────────────────────────────────────

    # CÁCH 1 (Khuyên dùng): Upload file kaggle.json
    # Tải kaggle.json từ kaggle.com/settings → API → Create New Token
    # rồi upload lên Colab
    from google.colab import files
    import json

    kaggle_dir = Path('/root/.kaggle')
    kaggle_dir.mkdir(exist_ok=True)

    kaggle_json = kaggle_dir / 'kaggle.json'
    if not kaggle_json.exists():
        print('📂 Upload file kaggle.json (tải từ kaggle.com/settings → API):')
        uploaded = files.upload()
        for fname, content in uploaded.items():
            kaggle_json.write_bytes(content)
        kaggle_json.chmod(0o600)
        print('✅ kaggle.json đã lưu!')
    else:
        print('✅ kaggle.json đã có sẵn')

    DATA_DIR = Path('/content/home_credit')

else:
    # LOCAL: đọc từ .env
    # Đảm bảo file .env đã có KAGGLE_USERNAME và KAGGLE_KEY
    from dotenv import load_dotenv
    ROOT_DIR = Path('..').resolve()
    load_dotenv(ROOT_DIR / '.env')

    if os.getenv('KAGGLE_USERNAME'):
        os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY']      = os.getenv('KAGGLE_KEY')

    DATA_DIR = ROOT_DIR / 'data' / 'raw' / 'home_credit'

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'📁 Thư mục dữ liệu: {DATA_DIR}')

In [ ]:
# Tải dữ liệu từ Kaggle
# Competition: home-credit-default-risk
# Lưu ý: cần accept competition rules trên Kaggle trước
# (vào https://www.kaggle.com/competitions/home-credit-default-risk → Join Competition)

MAIN_FILE = DATA_DIR / 'application_train.csv'

if not MAIN_FILE.exists():
    print('⬇️  Đang tải application_train.csv (~166 MB)...')
    os.system(f'kaggle competitions download -c home-credit-default-risk -f application_train.csv -p {DATA_DIR} --quiet')

    # Giải nén nếu cần
    zip_file = DATA_DIR / 'application_train.csv.zip'
    if zip_file.exists():
        import zipfile
        with zipfile.ZipFile(zip_file) as z:
            z.extractall(DATA_DIR)
        zip_file.unlink()

    print('✅ Tải xong!')
else:
    print(f'✅ File đã tồn tại: {MAIN_FILE.name} ({MAIN_FILE.stat().st_size/1e6:.1f} MB)')

# Tải thêm bureau.csv cho phân tích lịch sử tín dụng
BUREAU_FILE = DATA_DIR / 'bureau.csv'
if not BUREAU_FILE.exists():
    print('⬇️  Đang tải bureau.csv...')
    os.system(f'kaggle competitions download -c home-credit-default-risk -f bureau.csv -p {DATA_DIR} --quiet')
    zip_file = DATA_DIR / 'bureau.csv.zip'
    if zip_file.exists():
        import zipfile
        with zipfile.ZipFile(zip_file) as z:
            z.extractall(DATA_DIR)
        zip_file.unlink()
    print('✅ Tải xong!')
else:
    print(f'✅ File đã tồn tại: {BUREAU_FILE.name}')

## 2️⃣ Đọc & Xem tổng quan dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('husl')

print('📖 Đọc application_train.csv...')
df = pd.read_csv(MAIN_FILE, low_memory=False)
print(f'✅ Đọc xong: {df.shape[0]:,} dòng × {df.shape[1]} cột')

In [ ]:
# Tổng quan cơ bản
print('=' * 60)
print('TỔNG QUAN BẢNG application_train')
print('=' * 60)
print(f'Số dòng       : {df.shape[0]:>10,}')
print(f'Số cột        : {df.shape[1]:>10}')
print(f'Bộ nhớ dùng   : {df.memory_usage(deep=True).sum()/1e6:>9.1f} MB')
print(f'Tỷ lệ vỡ nợ  : {df["TARGET"].mean()*100:>9.2f}%')
print(f'  - Không vỡ nợ (0): {(df["TARGET"]==0).sum():>8,} ({(df["TARGET"]==0).mean()*100:.1f}%)')
print(f'  - Vỡ nợ (1)      : {(df["TARGET"]==1).sum():>8,} ({(df["TARGET"]==1).mean()*100:.1f}%)')
print()

# Phân loại cột
num_cols = df.select_dtypes(include='number').columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Cột số (numeric): {len(num_cols)}')
print(f'Cột phân loại   : {len(cat_cols)}')
df.head(3)

## 3️⃣ Phân tích Missing Values

In [ ]:
# Tính % missing cho từng cột
missing = (
    df.isnull()
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={'index': 'column', 0: 'missing_pct'})
    .sort_values('missing_pct', ascending=False)
)
missing = missing[missing['missing_pct'] > 0]

print(f'Số cột có missing: {len(missing)} / {df.shape[1]}')
print(f'Cột missing nhiều nhất:')
print(missing.head(15).to_string(index=False))

In [ ]:
# Biểu đồ: Top 20 cột missing nhiều nhất
top_missing = missing.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(
    top_missing['column'][::-1],
    top_missing['missing_pct'][::-1],
    color=sns.color_palette('Reds_r', len(top_missing))
)
ax.axvline(50, color='gray', linestyle='--', alpha=0.5, label='50% ngưỡng')
ax.set_xlabel('Tỷ lệ Missing (%)')
ax.set_title('Top 20 cột có nhiều giá trị thiếu nhất', fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())

# Thêm nhãn số %
for bar, val in zip(bars, top_missing['missing_pct'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=8)

plt.tight_layout()
plt.show()
print()

# Phân nhóm theo mức độ missing
print('Phân nhóm mức độ missing:')
bins = [0, 5, 20, 50, 100]
labels = ['< 5%', '5–20%', '20–50%', '> 50%']
missing['nhom'] = pd.cut(missing['missing_pct'], bins=bins, labels=labels)
print(missing.groupby('nhom', observed=True).size().to_string())

## 4️⃣ Phân tích Outlier — DAYS_EMPLOYED (Lỗi quan trọng!)

In [ ]:
# DAYS_EMPLOYED = 365243 là mã lỗi đặc biệt (không phải giá trị thật)
# Đây là phát hiện quan trọng cần ghi vào báo cáo!

days_emp = df['DAYS_EMPLOYED']
anomaly_count = (days_emp == 365243).sum()
anomaly_pct   = anomaly_count / len(df) * 100

print('=' * 55)
print('PHÂN TÍCH DAYS_EMPLOYED')
print('=' * 55)
print(f'Tổng số dòng         : {len(df):,}')
print(f'Giá trị = 365243     : {anomaly_count:,} ({anomaly_pct:.1f}%)')
print(f'  → Đây là mã lỗi: khách hàng không đi làm / hưu trí')
print(f'Phân vị 99%          : {days_emp[days_emp != 365243].quantile(0.99):.0f}')
print(f'Min (hợp lệ)         : {days_emp[days_emp != 365243].min():.0f}')
print(f'Max (hợp lệ)         : {days_emp[days_emp != 365243].max():.0f}')

# So sánh tỷ lệ vỡ nợ: có việc vs không có việc
df['has_employment'] = (days_emp != 365243).astype(int)
comp = df.groupby('has_employment')['TARGET'].agg(['mean','count'])
comp.index = ['Không việc làm (365243)', 'Có việc làm']
comp.columns = ['Tỷ lệ vỡ nợ', 'Số lượng']
comp['Tỷ lệ vỡ nợ'] = comp['Tỷ lệ vỡ nợ'].map('{:.2%}'.format)
print()
print('So sánh tỷ lệ vỡ nợ:')
print(comp.to_string())

In [ ]:
# Biểu đồ phân phối DAYS_EMPLOYED (sau khi loại 365243)
valid_emp = days_emp[days_emp != 365243].abs() / 365  # Đổi sang năm

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(valid_emp, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Số năm làm việc')
axes[0].set_ylabel('Số lượng')
axes[0].set_title('Phân phối Số năm làm việc\n(đã loại mã lỗi 365243)', fontweight='bold')

# Tỷ lệ vỡ nợ theo nhóm thâm niên
df['years_employed_clean'] = df.apply(
    lambda r: abs(r['DAYS_EMPLOYED']) / 365 if r['DAYS_EMPLOYED'] != 365243 else np.nan, axis=1
)
df['emp_group'] = pd.cut(df['years_employed_clean'], [0,1,3,5,10,50],
                          labels=['< 1 năm','1–3 năm','3–5 năm','5–10 năm','> 10 năm'])
emp_default = df.groupby('emp_group', observed=True)['TARGET'].mean() * 100

emp_default.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_xlabel('Thâm niên làm việc')
axes[1].set_ylabel('Tỷ lệ vỡ nợ (%)')
axes[1].set_title('Tỷ lệ vỡ nợ theo thâm niên', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.show()

## 5️⃣ Phân phối biến mục tiêu TARGET & Các yếu tố nhân khẩu học

In [ ]:
# Tạo cột phái sinh trước
df['age_years'] = (df['DAYS_BIRTH'].abs() / 365).astype(int)
df['age_group'] = pd.cut(df['age_years'], [0,25,35,45,55,100],
                          labels=['< 25','25–34','35–44','45–54','55+'])

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Tỷ lệ vỡ nợ theo các yếu tố nhân khẩu học', fontsize=14, fontweight='bold')

def plot_default_rate(col, ax, title, top_n=None):
    data = df.groupby(col, observed=True)['TARGET'].agg(['mean','count'])
    data.columns = ['rate','count']
    data = data[data['count'] >= 50]  # Lọc nhóm ít mẫu
    if top_n:
        data = data.nlargest(top_n, 'count')
    data['rate'] = data['rate'] * 100
    data.sort_values('rate', ascending=False)['rate'].plot(
        kind='bar', ax=ax, color='steelblue', edgecolor='white'
    )
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.tick_params(axis='x', rotation=30)
    ax.axhline(df['TARGET'].mean()*100, color='red', linestyle='--',
                alpha=0.6, label=f'TB: {df["TARGET"].mean()*100:.1f}%')
    ax.legend(fontsize=8)

plot_default_rate('age_group',          axes[0,0], 'Nhóm tuổi')
plot_default_rate('CODE_GENDER',        axes[0,1], 'Giới tính')
plot_default_rate('NAME_EDUCATION_TYPE',axes[0,2], 'Học vấn')
plot_default_rate('NAME_INCOME_TYPE',   axes[1,0], 'Loại thu nhập')
plot_default_rate('NAME_FAMILY_STATUS', axes[1,1], 'Tình trạng hôn nhân')
plot_default_rate('NAME_HOUSING_TYPE',  axes[1,2], 'Loại nhà ở')

plt.tight_layout()
plt.show()

## 6️⃣ Phân tích EXT_SOURCE — Điểm tín dụng bên ngoài

In [ ]:
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Phân phối EXT_SOURCE theo TARGET (0=không vỡ nợ, 1=vỡ nợ)',
             fontsize=12, fontweight='bold')

for i, col in enumerate(ext_cols):
    for target, color, label in [(0,'steelblue','Không vỡ nợ'), (1,'coral','Vỡ nợ')]:
        data = df[df['TARGET'] == target][col].dropna()
        axes[i].hist(data, bins=40, alpha=0.6, color=color, label=label, density=True)
    axes[i].set_xlabel('Điểm')
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend()

plt.tight_layout()
plt.show()

# Tương quan với TARGET
print('\nTương quan Pearson với TARGET:')
for col in ext_cols:
    corr = df[[col, 'TARGET']].dropna().corr().loc[col, 'TARGET']
    print(f'  {col}: {corr:+.4f}  (âm = điểm cao → ít vỡ nợ ✅)')

## 7️⃣ Phân tích Loan Amount & Income

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Phân phối các chỉ số tài chính', fontsize=13, fontweight='bold')

def plot_kde_by_target(col, ax, title, xlim=None):
    for target, color, label in [(0,'steelblue','Không vỡ nợ'), (1,'coral','Vỡ nợ')]:
        data = df[df['TARGET'] == target][col].dropna()
        if xlim:
            data = data[data <= xlim]
        data.plot.kde(ax=ax, color=color, label=label)
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.set_xlabel('')

plot_kde_by_target('AMT_CREDIT',       axes[0,0], 'Số tiền vay (AMT_CREDIT)',    xlim=3e6)
plot_kde_by_target('AMT_INCOME_TOTAL', axes[0,1], 'Thu nhập (AMT_INCOME_TOTAL)', xlim=5e5)
plot_kde_by_target('AMT_ANNUITY',      axes[1,0], 'Trả góp tháng (AMT_ANNUITY)', xlim=1e5)

# Scatter: Income vs Credit
sample = df.sample(3000, random_state=42)
axes[1,1].scatter(
    sample['AMT_INCOME_TOTAL'] / 1000,
    sample['AMT_CREDIT'] / 1000,
    c=sample['TARGET'].map({0:'steelblue', 1:'coral'}),
    alpha=0.4, s=15
)
axes[1,1].set_xlabel('Thu nhập (K)')
axes[1,1].set_ylabel('Số tiền vay (K)')
axes[1,1].set_title('Thu nhập vs Số tiền vay\n(🔵=OK, 🟠=vỡ nợ)', fontweight='bold')
axes[1,1].set_xlim(0, 500)
axes[1,1].set_ylim(0, 3000)

plt.tight_layout()
plt.show()

## 8️⃣ Correlation Heatmap — Top cột quan trọng

In [ ]:
# Chọn các cột số quan trọng để xem tương quan
key_cols = [
    'TARGET',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
    'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'age_years',
    'REGION_RATING_CLIENT',
    'CNT_CHILDREN',
]

corr_matrix = df[key_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f', center=0,
    cmap='RdBu_r', vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    ax=ax
)
ax.set_title('Ma trận tương quan (các biến chính)', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Top biến tương quan với TARGET
print('\nTop 10 biến tương quan mạnh nhất với TARGET:')
print(corr_matrix['TARGET'].drop('TARGET').abs().sort_values(ascending=False).head(10).to_string())

## 9️⃣ Báo cáo Tự động với ydata-profiling

In [ ]:
# ydata-profiling tự động tạo báo cáo HTML đầy đủ
# Chạy cell này sẽ mất 3–10 phút, output là file HTML mở được trong browser

RUN_PROFILING = True  # ← Đổi thành False nếu muốn bỏ qua bước này

if RUN_PROFILING:
    from ydata_profiling import ProfileReport

    print('🔄 Đang tạo báo cáo profiling (3–10 phút)...')

    # Chỉ lấy 50K dòng để profiling nhanh hơn
    df_sample = df.sample(min(50_000, len(df)), random_state=42)

    profile = ProfileReport(
        df_sample,
        title='CCIP — Home Credit Data Profile',
        explorative=True,
        # Tắt các phép tính nặng để chạy nhanh hơn
        correlations={'auto': {'calculate': True}},
        missing_diagrams={'heatmap': False, 'dendrogram': False},
    )

    if IN_COLAB:
        # Trên Colab: hiển thị inline
        profile.to_notebook_iframe()
    else:
        # Trên Local: lưu ra file HTML
        out_path = Path('..') / 'docs' / 'data_profile_report.html'
        out_path.parent.mkdir(exist_ok=True)
        profile.to_file(out_path)
        print(f'✅ Báo cáo lưu tại: {out_path.resolve()}')
        print('   Mở file HTML này bằng trình duyệt để xem!')
else:
    print('ℹ️  Bỏ qua profiling. Đổi RUN_PROFILING = True để chạy.')

## 🏁 Tóm tắt phát hiện (cập nhật sau khi phân tích)

| # | Phát hiện | Hành động xử lý trong staging |
|---|-----------|-------------------------------|
| 1 | `DAYS_EMPLOYED = 365243` (~18% dòng) — mã lỗi | → Chuyển thành `NULL`, tạo cột `has_employment` |
| 2 | `EXT_SOURCE_1/2/3` có 30–56% missing | → Điền bằng trung vị, tạo `ext_score_avg` |
| 3 | `AMT_INCOME_TOTAL` có outlier rất cao | → Winsorize tại p99 trong staging |
| 4 | Mất cân bằng nhãn: 92% không vỡ nợ, 8% vỡ nợ | → Ghi chú trong báo cáo; dùng stratified split |
| 5 | `EXT_SOURCE_2` tương quan âm mạnh nhất với TARGET | → Ưu tiên trong phân tích |

---
> **Bước tiếp theo:** Chạy `load_raw.py` → `stg_application.sql` để nạp và làm sạch dữ liệu này vào PostgreSQL